<a href="https://colab.research.google.com/github/toche7/BasicAIAgentColab/blob/main/BasicHarness2_Tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent: Tool Calling

### ส่วนที่ 1: การกำหนดเครื่องมือ (Tool Definitions)

ในเซลล์นี้จะมีการกำหนดฟังก์ชันหลัก 2 ฟังก์ชันที่จะใช้เป็นเครื่องมือ (Tools) ให้กับ AI:
1. `calc`: ฟังก์ชันสำหรับคำนวณคณิตศาสตร์อย่างปลอดภัย โดยใช้ Regex เพื่อกรองเฉพาะตัวเลขและเครื่องหมายที่อนุญาต
2. `read_file`: ฟังก์ชันสำหรับการอ่านไฟล์ข้อความจากโฟลเดอร์ทำงานอย่างปลอดภัย

นอกจากนี้ยังมีการกำหนดตัวแปร `SCHEMAS` เพื่อใช้อธิบายโครงสร้างและวิธีเรียกใช้งานฟังก์ชันเหล่านี้ให้โมเดล AI เข้าใจ

In [ ]:
"""  Part 1 - Tool Definitions"""
import json  # Used for parsing and formatting JSON payloads for tool arguments
import re  # Used for secure input validation of arithmetic expressions
from pathlib import Path  # Used for safe, cross-platform file path resolution

def calc(expression: str) -> str:
    """Evaluate an arithmetic expression securely by limiting allowed characters."""
    # Only allow numbers, spaces, parentheses, and standard arithmetic operators
    if not re.match(r'^[0-9+\-*/().\s]+$', expression):
        raise ValueError("only numbers and + - * / are allowed")
    try:
        # Safe to evaluate since we restricted character set completely
        return str(eval(expression, {"__builtins__": None}, {}))
    except Exception as e:
        raise ValueError(f"Invalid math expression: {e}")

def read_file(path: str) -> str:
    """Read a local file safely within the working directory."""
    p = Path(path).resolve()
    if not p.is_relative_to(Path.cwd().resolve()):
        raise PermissionError("path is outside the working directory")
    return p.read_text()[:8000]


# Tool schemas to pass to the model
SCHEMAS = [
    {"type": "function", "function": {
        "name": "calc", "description": "Evaluate an arithmetic expression like '12 * (3 + 4)'.",
        "parameters": {"type": "object", "properties": {"expression": {"type": "string"}},
                       "required": ["expression"]}}},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a text file in the working directory.",
        "parameters": {"type": "object", "properties": {"path": {"type": "string"}},
                       "required": ["path"]}}},
]


### การแมปและการประมวลผลเครื่องมือ

เซลล์นี้สร้างพจนานุกรม (Dictionary) ชื่อ `FUNCTIONS` เพื่อเก็บการอ้างอิงของฟังก์ชัน และฟังก์ชัน `execute` สำหรับแปลงอาร์กิวเมนต์ที่ได้จาก AI (ซึ่งเป็น JSON string) แล้วเรียกใช้ฟังก์ชันจริงเพื่อคืนค่ากลับไป

In [ ]:

# Mapping of tool names to their respective functions
FUNCTIONS = {"calc": calc, "read_file": read_file}


def execute(name: str, arguments: str) -> str:
    """Execute a specified tool with arguments."""
    try:
        return FUNCTIONS[name](**json.loads(arguments or "{}"))
    except Exception as e:
        return f"Error: {type(e).__name__}: {e}"

### ส่วนที่ 2: การสนทนาและการเรียกใช้งาน (Chat and Runtime Execution)

เซลล์นี้ทำหน้าที่ตั้งค่าการเชื่อมต่อกับ Groq API โดยกำหนด URL, ตัวเลือกของโมเดล และสร้างฟังก์ชัน `chat` สำหรับส่งข้อความพร้อมโครงสร้างเครื่องมือ (Tools) ไปยังโมเดลผ่านทาง HTTP POST request

In [ ]:
"""Part 2 - Chat and Runtime execution"""
import os
import sys
import httpx
from google.colab import userdata


URL = "https://api.groq.com/openai/v1/chat/completions"
MODEL = "qwen/qwen3.8-27b"

try:
    api_key = userdata.get('GROQ_API_KEY')
except Exception:
    api_key = os.environ.get('GROQ_API_KEY')

def chat(messages: list[dict], tools: list[dict] | None = None) -> dict:
    """POST the conversation with tools configuration to Groq API."""
    if not api_key:
        raise ValueError("GROQ_API_KEY not found.")

    body = {"model": MODEL, "messages": messages, "max_tokens": 800}
    if tools:
        body["tools"] = tools

    headers = {"Authorization": f"Bearer {api_key}"}
    resp = httpx.post(URL, headers=headers, json=body, timeout=60)
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]


### การตั้งคำถามและการส่งข้อความครั้งแรก

ในเซลล์นี้เราจะกำหนดคำถามของฝั่งผู้ใช้ เช่น `"2 plus 3"` และเริ่มส่งคำขอแรกไปยังโมเดลพร้อมโครงสร้างเครื่องมือ (`SCHEMAS`) เพื่อให้โมเดลตัดสินใจว่าจะใช้เครื่องมือใด

In [ ]:
question = "read file note.txt, 2 plus 3 use tool calling."
messages = [{"role": "user", "content": question}]

# First call
reply = chat(messages, SCHEMAS)
calls = reply.get("tool_calls") or []

### ตรวจสอบผลลัพธ์ที่ตอบกลับจากโมเดล

แสดงค่าของวัตถุ `reply` ที่ได้จาก API เพื่อดูรูปแบบข้อมูลที่โมเดลตอบกลับมา

In [ ]:
reply

### แสดงรายการการเรียกใช้เครื่องมือ (Tool Calls)

ตรวจสอบรายการ `calls` ซึ่งเก็บข้อมูลเฉพาะส่วนที่โมเดลร้องขอให้เรียกใช้เครื่องมือภายนอก

In [ ]:
calls

### ประมวลผลและส่งผลลัพธ์กลับไปยัง AI

ตรวจสอบว่ามีการร้องขอให้เรียกใช้เครื่องมือหรือไม่:
* หากไม่มี: จะแสดงข้อความตอบกลับทั่วไป
* หากมี: จะทำการประมวลผลฟังก์ชันเครื่องมือนั้น ๆ นำผลลัพธ์กลับมาใส่ในประวัติการสนทนา แล้วส่งกลับไปยังโมเดลอีกครั้งเป็นครั้งที่สอง เพื่อให้ได้คำตอบสุดท้ายที่เป็นธรรมชาติ

In [ ]:

if not calls:
     print(reply["content"])
else:
     messages.append({"role": "assistant", "content": reply.get("content") or "", "tool_calls": calls})
     for call in calls:
        fn = call["function"]
        print(f"[tool] {fn['name']}({fn['arguments']})")
        messages.append({"role": "tool", "tool_call_id": call["id"],
                         "content": execute(fn["name"], fn["arguments"])})
     # Second call, feeding back the tool results
     print(chat(messages)["content"])